# Holly Exit Optimizer - Interactive Analysis Notebook

Ad-hoc analysis of 28,875 trades across 134 strategies, 352 sectors, and 10 years.

**Columns available**: 96 enriched features including probability engine, regime, sector, TOD, rolling metrics.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from scipy import stats

DATA = Path('../holly_exit/output/holly_analytics.csv')
df = pd.read_csv(DATA)
df['entry_dt'] = pd.to_datetime(df['entry_time'])
df['holly_pnl'] = df['holly_pnl'].fillna(0)
df = df.sort_values('entry_dt').reset_index(drop=True)

print(f'{len(df):,} trades | {df["trade_year"].nunique()} years | {df["strategy"].nunique()} strategies | {df.shape[1]} columns')

## Quick Stats

In [ ]:
print(f"Win Rate:      {df['is_winner'].mean()*100:.1f}%")
print(f"Avg PnL:       ${df['holly_pnl'].mean():,.0f}")
print(f"Total PnL:     ${df['holly_pnl'].sum():,.0f}")
print(f"Profit Factor: {df.loc[df['holly_pnl']>0,'holly_pnl'].sum() / abs(df.loc[df['holly_pnl']<=0,'holly_pnl'].sum()):.2f}")
print(f"Sharpe:        {df['holly_pnl'].mean() / df['holly_pnl'].std() * np.sqrt(252):.1f}")
print(f"Date Range:    {df['entry_dt'].min():%Y-%m-%d} to {df['entry_dt'].max():%Y-%m-%d}")

## Conditional Probability Trees (IF -> THEN)

The core "if-then" analysis: what's the win rate given specific conditions?

In [ ]:
def conditional_wr(*filters, min_n=20):
    """Compute WR for trades matching ALL given filters.
    
    Usage:
      conditional_wr(df['strategy'] == 'Breakdown Short')
      conditional_wr(df['trend_regime'] == 'downtrend', df['tod_bucket'] == '07:00')
      conditional_wr(df['strategy'] == 'Breakdown Short', df['trend_regime'] == 'sideways', df['tod_bucket'].isin(['07:00','07:30']))
    """
    mask = pd.Series([True] * len(df), index=df.index)
    for f in filters:
        mask = mask & f
    sub = df.loc[mask]
    n = len(sub)
    if n < min_n:
        return f'Insufficient data ({n} trades, need {min_n})'
    wr = sub['is_winner'].mean() * 100
    avg = sub['holly_pnl'].mean()
    t, p = stats.ttest_1samp(sub['holly_pnl'], 0)
    return f'n={n:,} | WR={wr:.1f}% | avg=${avg:,.0f} | t={t:.1f} | p={p:.4f}'

# Examples:
print('Breakdown Short + downtrend + morning:')
print(conditional_wr(
    df['strategy'] == 'Breakdown Short',
    df['trend_regime'] == 'downtrend',
    df['tod_bucket'].isin(['07:00', '07:30'])
))

print('\nAll strategies + sideways regime:')
print(conditional_wr(df['trend_regime'] == 'sideways'))

In [ ]:
# Full conditional probability matrix: Strategy x Regime
def cond_matrix(dim1, dim2, min_n=20):
    """Cross-tabulation of win rates for two dimensions."""
    valid = df.dropna(subset=[dim1, dim2])
    ct = valid.groupby([dim1, dim2]).agg(
        n=('holly_pnl', 'count'),
        wr=('is_winner', 'mean'),
        avg_pnl=('holly_pnl', 'mean'),
    ).reset_index()
    ct = ct[ct['n'] >= min_n]
    ct['wr'] = (ct['wr'] * 100).round(1)
    ct['avg_pnl'] = ct['avg_pnl'].round(0)
    pivot = ct.pivot_table(index=dim1, columns=dim2, values='wr')
    return pivot

# Strategy x Regime heatmap
pivot = cond_matrix('strategy', 'trend_regime', min_n=30)
fig = px.imshow(pivot, text_auto='.0f', color_continuous_scale='RdYlGn',
                zmin=35, zmax=70, aspect='auto',
                title='Win Rate (%) by Strategy x Regime (min 30 trades)')
fig.update_layout(height=600, width=900)
fig.show()

## Equity Curves by Filter Layer

In [ ]:
MIN_SECTOR_TRADES = 50

configs = {
    'Baseline': pd.Series([True]*len(df), index=df.index),
    '+Edge Verdict': df['prob_edge_verdict'] == 'Strong Edge',
    '+TOD Window': df['tod_bucket'].isin(['06:30','07:00','07:30','08:00']),
    '+Regime': df['trend_regime'].isin(['sideways','downtrend']),
    '+Sector (robust)': (df['sector_win_rate'] > 0.52) & (df['sector_trades'] >= MIN_SECTOR_TRADES),
}

fig = go.Figure()
running = pd.Series([True]*len(df), index=df.index)
colors = ['#94a3b8', '#3b82f6', '#f59e0b', '#8b5cf6', '#10b981']

for (name, filt), color in zip(configs.items(), colors):
    if name != 'Baseline':
        running = running & filt
    sub = df.loc[running].sort_values('entry_dt')
    wr = sub['is_winner'].mean() * 100
    fig.add_trace(go.Scatter(
        x=sub['entry_dt'], y=sub['holly_pnl'].cumsum(),
        mode='lines', name=f'{name} (n={len(sub):,}, WR={wr:.1f}%)',
        line=dict(color=color, width=2),
    ))

fig.update_layout(template='plotly_white', height=500,
                  title='Cumulative Filter Stacking — Equity Curves',
                  yaxis=dict(tickformat='$,.0f'))
fig.show()

## Feature Ablation: Which Layer Adds the Most Edge?

In [ ]:
# Apply each filter INDEPENDENTLY and measure impact
indep_filters = {
    'Edge Verdict (Strong)': df['prob_edge_verdict'] == 'Strong Edge',
    'TOD (06:30-08:00)': df['tod_bucket'].isin(['06:30','07:00','07:30','08:00']),
    'Regime (sideways/down)': df['trend_regime'].isin(['sideways','downtrend']),
    'Sector (robust)': (df['sector_win_rate'] > 0.52) & (df['sector_trades'] >= MIN_SECTOR_TRADES),
}

baseline_wr = df['is_winner'].mean() * 100
results = []
for name, mask in indep_filters.items():
    sub = df.loc[mask]
    wr = sub['is_winner'].mean() * 100
    results.append({
        'Filter': name,
        'Trades': len(sub),
        '% Retained': round(len(sub)/len(df)*100, 1),
        'WR%': round(wr, 1),
        'WR Lift (pp)': round(wr - baseline_wr, 1),
        'Avg PnL': round(sub['holly_pnl'].mean(), 0),
    })

pd.DataFrame(results).sort_values('WR Lift (pp)', ascending=False)

## Custom Analysis Template

Use this cell for your own ad-hoc queries:

In [ ]:
# Template: groupby anything and measure edge
# Modify dim and filter_mask to explore any hypothesis

dim = 'strategy'  # Change to: sector, trend_regime, tod_bucket, trade_year, direction, etc.
filter_mask = df['trend_regime'] == 'downtrend'  # Change to any condition

analysis = df.loc[filter_mask].groupby(dim).agg(
    n=('holly_pnl', 'count'),
    wr=('is_winner', 'mean'),
    avg_pnl=('holly_pnl', 'mean'),
    total_pnl=('holly_pnl', 'sum'),
    median_pnl=('holly_pnl', 'median'),
).reset_index()
analysis['wr'] = (analysis['wr'] * 100).round(1)
analysis = analysis[analysis['n'] >= 20]
analysis.sort_values('wr', ascending=False).head(20)

## Walk-Forward Validation

Check if any edge persists out-of-sample:

In [ ]:
def walk_forward(mask, n_folds=3):
    """Walk-forward OOS validation for a filter mask."""
    sub = df.loc[mask].sort_values('entry_dt')
    fold_size = len(sub) // n_folds
    results = []
    for i in range(1, n_folds):
        train = sub.iloc[:fold_size*i]
        test = sub.iloc[fold_size*i:fold_size*(i+1)]
        results.append({
            'fold': i,
            'train_n': len(train),
            'test_n': len(test),
            'train_wr': round(train['is_winner'].mean()*100, 1),
            'test_wr': round(test['is_winner'].mean()*100, 1),
            'decay_pp': round((test['is_winner'].mean() - train['is_winner'].mean())*100, 1),
        })
    return pd.DataFrame(results)

# Full stack walk-forward
full_mask = (
    (df['prob_edge_verdict'] == 'Strong Edge') &
    df['tod_bucket'].isin(['06:30','07:00','07:30','08:00']) &
    df['trend_regime'].isin(['sideways','downtrend']) &
    (df['sector_win_rate'] > 0.52) & (df['sector_trades'] >= MIN_SECTOR_TRADES)
)
print('Full Stack Walk-Forward:')
walk_forward(full_mask)

## Available Columns Reference

In [ ]:
groups = {
    'Trade Core': [c for c in df.columns if c in ['trade_id','symbol','strategy','direction','entry_time','exit_time','entry_price','exit_price','holly_pnl','shares','stop_price','target_price','mfe','mae']],
    'Time': [c for c in df.columns if c in ['trade_date','trade_year','trade_month','trade_dow','entry_hour','tod_bucket']],
    'Regime': [c for c in df.columns if 'regime' in c or c in ['sma20','sma5','trend_slope','above_sma20','atr14','atr_pct','daily_range_pct','rsi14','roc5','roc20']],
    'Sector/Ticker': [c for c in df.columns if c in ['company_name','sic_code','sector','primary_exchange','market_cap','total_employees','address_state','ipo_date']],
    'Conditional': [c for c in df.columns if '_cond_' in c or 'sector_' in c or '_rolling_' in c],
    'Probability': [c for c in df.columns if 'prob_' in c],
    'Optimization': [c for c in df.columns if 'opt_' in c],
}
for group, cols in groups.items():
    print(f'\n{group} ({len(cols)} cols):')
    for c in cols:
        print(f'  {c:<30} {df[c].dtype}  ({df[c].notna().sum():,} non-null)')